# BTC-USD: will the next 5 days be up or down?
MLP classifier adapted from the WQU AAPL market-timing exercise.

Change `HORIZON` and `LOOKBACKS` in the first cell to experiment. Everything else follows from them.

In [ ]:
import json
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn import metrics
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

TICKER    = "BTC-USD"
HORIZON   = 5                    # predict the sign of the NEXT 5 days' return
LOOKBACKS = [3, 5, 10, 20, 60]   # input windows (days of past return fed to the model)

In [ ]:
raw = yf.download(TICKER, start="2015-01-01", interval="1d", auto_adjust=True, progress=False)

# yfinance can return two-level columns (Price, Ticker): flatten to one level
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)
if raw.index.tz is not None:
    raw.index = raw.index.tz_localize(None)
raw = raw.sort_index()

# BTC trades 24/7, so today's daily bar is still forming. Drop it so we only use completed closes.
today_utc = pd.Timestamp.now(tz="UTC").tz_localize(None).normalize()
raw = raw[raw.index < today_utc]

raw.tail(3)

## Features and target
Same idea as the class notebook, but the label is now the sign of the next `HORIZON` days' compounded return.
Rows whose future is not known yet are removed **before** the 0/1 label is created (otherwise `NaN > 0` silently becomes 0).

In [ ]:
df = pd.DataFrame({"Ret": raw["Close"].pct_change() * 100}).dropna()

def compound(x):
    """Compound a window of % returns into one % return."""
    return 100 * (np.prod(1 + x / 100) - 1)

for n in LOOKBACKS:
    df[f"Ret{n}_i"] = df["Ret"].rolling(n).apply(compound, raw=True)
feature_cols = [f"Ret{n}_i" for n in LOOKBACKS]

# Return over t+1 ... t+HORIZON, lined up on row t
fwd = df["Ret"].rolling(HORIZON).apply(compound, raw=True).shift(-HORIZON)

df = df.dropna(subset=feature_cols)
fwd = fwd.loc[df.index]

# Latest completed day: features are known, outcome is not. This is the row you predict on live.
live_row = df[feature_cols].iloc[[-1]].astype("float32")

df["Output"] = (fwd > 0).astype(int)
df = df[fwd.notna()]            # keep only rows whose outcome is known

print(df.shape, "| share of 'up' labels:", round(df["Output"].mean(), 3))
df.tail()

In [ ]:
ts = int(0.2 * len(df))              # 20% test, chronological
split_time = len(df) - ts
test_time = df.index[split_time:]
Ret_vector = df["Ret"].iloc[split_time:].values

X = df[feature_cols].astype("float32")
y = df["Output"].astype("float32")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=ts, shuffle=False)

# The last HORIZON training labels look into the test period; drop them to avoid leakage
X_train, y_train = X_train.iloc[:-HORIZON], y_train.iloc[:-HORIZON]
print(X_train.shape, X_test.shape)

In [ ]:
tf.keras.backend.clear_session()
tf.random.set_seed(1234)

n_dropout = 0.2
model = tf.keras.models.Sequential([
    tf.keras.layers.Dense(25, activation="relu"),
    tf.keras.layers.Dropout(n_dropout),
    tf.keras.layers.Dense(15, activation="relu"),
    tf.keras.layers.Dropout(n_dropout),
    tf.keras.layers.Dense(10, activation="relu"),
    tf.keras.layers.Dropout(n_dropout),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
              loss="binary_crossentropy", metrics=["accuracy"])

es = tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", mode="max", patience=20,
                                      restore_best_weights=True, verbose=1)
class_weight = {0: (float(np.mean(y_train)) / 0.5) * 1.2, 1: 1.0}
print(class_weight)

In [ ]:
history = model.fit(X_train, y_train, validation_split=0.2, epochs=500, batch_size=32,
                    verbose=0, callbacks=[es], class_weight=class_weight)
print("Training stopped after", len(history.history["loss"]), "epochs")

## Predictive performance
Compare accuracy with the *always up* baseline: that is what you get for free by never thinking.

In [ ]:
y_prob = model.predict(X_test, verbose=0)
y_pred = (y_prob > 0.5).astype(int).ravel()

print("Test accuracy        :", round(metrics.accuracy_score(y_test, y_pred), 3))
print("'Always up' baseline :", round(float(y_test.mean()), 3))

ConfusionMatrixDisplay(metrics.confusion_matrix(y_test, y_pred), display_labels=["Down", "Up"]).plot()
plt.show()

## Backtest
The signal at the close of day *t* is traded during day *t+1* (`shift(1)`), so there is no look-ahead.
`COST` charges a fee/slippage every time the position changes, which matters far more at a 5-day horizon than at 120 days.

In [ ]:
COST = 0.001   # 0.1% per unit of position change (fees + slippage). Set this to your real venue costs.

bt = pd.DataFrame({"Pred": y_pred, "Ret": Ret_vector / 100}, index=test_time)
bt["Pos_LS"] = np.where(bt["Pred"] == 1, 1, -1)    # long/short
bt["Pos_L"]  = bt["Pos_LS"].clip(lower=0)          # long / cash

def strat_returns(pos, ret, cost):
    pos = pos.shift(1).fillna(0)                   # signal at close of t is held during t+1
    gross = pos * ret
    net = gross - cost * pos.diff().abs().fillna(0)
    return gross, net

bt["LS_gross"], bt["LS_net"] = strat_returns(bt["Pos_LS"], bt["Ret"], COST)
bt["L_gross"],  bt["L_net"]  = strat_returns(bt["Pos_L"],  bt["Ret"], COST)

cum = (1 + bt[["LS_gross", "LS_net", "L_gross", "L_net", "Ret"]]).cumprod() - 1
cum = cum.rename(columns={"Ret": "Buy&Hold"})

print((cum.iloc[-1] * 100).round(1).to_string())
print("Position changes (long/short):", int(bt["Pos_LS"].diff().abs().gt(0).sum()))

cum.plot(figsize=(12, 6), grid=True)
plt.ylabel("Cumulative return")
plt.show()

## Save the model and the settings the live script needs

In [ ]:
model.save("btc_5d_mlp.keras")
with open("btc_5d_config.json", "w") as f:
    json.dump({"ticker": TICKER, "horizon": HORIZON, "lookbacks": LOOKBACKS, "features": feature_cols}, f)

# What would the model say for the latest completed day?
p = float(model.predict(live_row, verbose=0)[0, 0])
print(live_row.index[0].date(), "->", "LONG" if p > 0.5 else "SHORT", f"(prob up = {p:.2f})")